# Notebook 03c v2 — UNSW Dirichlet calibration (parallel comparison)

**What this notebook does:**

Fits Dirichlet ODIR calibrator (Kull et al. 2019) per model on UNSW calibration set, applies to test set, compares against pre-calibration baseline AND the hybrid Platt/isotonic results (from the parallel UNSW hybrid notebook — if that hasn't run yet, the comparison table will skip hybrid columns).

**Why UNSW is the critical dataset for Dirichlet:**

Unlike NSL-KDD, UNSW-NB15's calibration set and test set have nearly identical class proportions for all 5 classes. This means distribution shift is minimal — if any calibration method can demonstrably improve probabilistic accuracy, it should be here. UNSW serves as the cleanest empirical test of calibration value.

**Predicted outcome (to be verified):**

- Dirichlet should improve Brier and ECE on most classes (because there's no distribution shift to defeat the calibrator)
- The improvement margin should be larger than on NSL
- If Dirichlet does NOT improve here, then post-hoc calibration is not effective on these IDS tasks regardless of method

## 1. Setup

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [6]:
!pip install -q dirichletcal

from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator
print('✓ dirichletcal package loaded')

✓ dirichletcal package loaded


In [7]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.metrics import brier_score_loss
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASET = 'unsw_nb15_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PREDS_DIR = MODELS_DIR / 'predictions'
PROBS_DIR = MODELS_DIR / 'probabilities'   # UNSW splits predictions and probabilities into separate folders
CALIB_DIR_HYBRID = Path(REPO) / 'calibrators' / DATASET
CALIB_DIR_DIRICHLET = Path(REPO) / 'calibrators' / f'{DATASET}_dirichlet'
CALIB_DIR_DIRICHLET.mkdir(parents=True, exist_ok=True)

ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Probabilities source: {PROBS_DIR}')
print(f'Dirichlet outputs dir: {CALIB_DIR_DIRICHLET}')

Dataset: unsw_nb15_v2
Probabilities source: /content/drive/MyDrive/XIDS_Research/xids-research/models/unsw_nb15_v2/probabilities
Dirichlet outputs dir: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/unsw_nb15_v2_dirichlet


## 2. Load labels and verify the no-shift hypothesis

In [9]:
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)

print(f'class_mappings.json keys: {list(class_info.keys())}')

# Handle different schemas: NSL uses 'multiclass_5', UNSW uses 'five_class_id_to_name'
mapping_5 = None
for key_candidate in ['multiclass_5', 'five_class_id_to_name', '5class', 'five_class', 'multiclass']:
    if key_candidate in class_info:
        mapping_5 = class_info[key_candidate]
        print(f"Using key: '{key_candidate}'")
        break

if mapping_5 is None:
    print(f'\nFull JSON content:')
    print(json.dumps(class_info, indent=2))
    raise KeyError("Could not find 5-class mapping. Update the candidate list.")

# Map is id->name (string keys "0".."4" -> class names) for both NSL and UNSW
INT_TO_CATEGORY = {int(k): v for k, v in mapping_5.items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'\nCalibration set: {len(y_calib_b):,}, Test set: {len(y_test_b):,}')
print(f'5-class names: {CLASS_NAMES_5}')
print()

# Class prior comparison
calib_counts = Counter(y_calib_5)
test_counts = Counter(y_test_5)
n_calib = len(y_calib_5)
n_test = len(y_test_5)

rows = []
for c in range(5):
    p_calib = calib_counts[c] / n_calib
    p_test = test_counts[c] / n_test
    rows.append({
        'Class': CLASS_NAMES_5[c],
        'n_calib': calib_counts[c],
        'p_calib (%)': round(100 * p_calib, 2),
        'n_test': test_counts[c],
        'p_test (%)': round(100 * p_test, 2),
        'p_test/p_calib': round(p_test / p_calib, 2) if p_calib > 0 else float('inf'),
    })

df = pd.DataFrame(rows)
print('UNSW class-prior comparison: calibration set vs test set')
print('=' * 90)
print(df.to_string(index=False))
print('=' * 90)
print()
print('Compare with NSL ratios: R2L 16.20x, U2R 7.49x (severe shift)')
print('UNSW ratios near 1.0 → minimal distribution shift, fair test of calibration efficacy')

class_mappings.json keys: ['binary', 'five_class_id_to_name', 'five_class_name_to_id', 'unsw_to_five_class', 'dropped_categories']
Using key: 'five_class_id_to_name'

Calibration set: 27,069, Test set: 63,461
5-class names: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

UNSW class-prior comparison: calibration set vs test set
 Class  n_calib  p_calib (%)  n_test  p_test (%)  p_test/p_calib
Normal    11200        41.38   37000       58.30            1.41
   DoS     2453         9.06    4089        6.44            0.71
 Probe     2498         9.23    4173        6.58            0.71
   R2L    10665        39.40   17777       28.01            0.71
   U2R      253         0.93     422        0.66            0.71

Compare with NSL ratios: R2L 16.20x, U2R 7.49x (severe shift)
UNSW ratios near 1.0 → minimal distribution shift, fair test of calibration efficacy


## 3. Helper functions

In [10]:
def expected_calibration_error(probs, labels, n_bins=10):
    """ECE on one-vs-rest binary probabilities."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        bin_conf = probs[mask].mean()
        bin_acc = labels[mask].mean()
        weight = mask.sum() / n
        ece += weight * abs(bin_conf - bin_acc)
    return float(ece)

def fit_dirichlet(p_calib_2d, y_calib, reg_lambda=1e-3):
    """Fit ODIR (regularised Dirichlet) calibrator."""
    cal = FullDirichletCalibrator(reg_lambda=reg_lambda, reg_mu=None)
    cal.fit(p_calib_2d, y_calib)
    return cal

def evaluate_calibrator(p_test_cal, y_test, n_classes):
    """Per-class Brier and ECE."""
    brier = {}
    ece = {}
    for c in range(n_classes):
        y_c = (y_test == c).astype(int)
        p_c = p_test_cal[:, c]
        brier[c] = float(brier_score_loss(y_c, p_c))
        ece[c] = expected_calibration_error(p_c, y_c, ECE_N_BINS)
    return brier, ece

print('✓ Helpers loaded')

✓ Helpers loaded


## 4. Calibrate all 9 models with Dirichlet

In [ ]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task}) with Dirichlet ODIR...')

    # UNSW: probability files live in probabilities/ subfolder
    p_calib = np.load(PROBS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PROBS_DIR / f'{name}_test_proba.npy')

    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])

    n_classes = p_calib.shape[1]
    y_calib_use = y_calib_b if task == 'binary' else y_calib_5
    y_test_use = y_test_b if task == 'binary' else y_test_5

    brier_pre, ece_pre = evaluate_calibrator(p_test, y_test_use, n_classes)

    try:
        cal = fit_dirichlet(p_calib, y_calib_use)
        p_test_cal = cal.predict_proba(p_test)
        success = True
    except Exception as e:
        print(f'  ✗ Dirichlet fit failed: {e}')
        print(f'    Falling back to identity for this model.')
        p_test_cal = p_test.copy()
        success = False

    brier_post, ece_post = evaluate_calibrator(p_test_cal, y_test_use, n_classes)

    np.save(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy', p_test_cal)

    if task == '5class':
        for c in range(5):
            cname = CLASS_NAMES_5[c]
            delta_b = brier_post[c] - brier_pre[c]
            arrow = '↓' if delta_b < 0 else ('↑' if delta_b > 0 else '−')
            print(f'  {cname:8s}: Brier {brier_pre[c]:.4f} → {brier_post[c]:.4f} {arrow}'
                  f'   ECE {ece_pre[c]:.4f} → {ece_post[c]:.4f}')
    else:
        delta_b = brier_post[1] - brier_pre[1]
        arrow = '↓' if delta_b < 0 else ('↑' if delta_b > 0 else '−')
        print(f'  Attack  : Brier {brier_pre[1]:.4f} → {brier_post[1]:.4f} {arrow}'
              f'   ECE {ece_pre[1]:.4f} → {ece_post[1]:.4f}')

    calibration_summary[name] = {
        'task': task,
        'method': 'dirichlet_odir' if success else 'failed_identity',
        'success': success,
        'brier_pre':  {int(k): v for k, v in brier_pre.items()},
        'brier_post': {int(k): v for k, v in brier_post.items()},
        'ece_pre':  {int(k): v for k, v in ece_pre.items()},
        'ece_post': {int(k): v for k, v in ece_post.items()},
    }

with open(CALIB_DIR_DIRICHLET / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated. Summary: {CALIB_DIR_DIRICHLET}/calibration_summary.json')


Calibrating rf_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.0951 → 0.1034 ↑   ECE 0.1018 → 0.1171

Calibrating xgb_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.0967 → 0.1061 ↑   ECE 0.1064 → 0.1208

Calibrating dnn_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.0984 → 0.1130 ↑   ECE 0.1059 → 0.1290

Calibrating rf_5class_smote (5class) with Dirichlet ODIR...
  Normal  : Brier 0.1115 → 0.1041 ↓   ECE 0.1272 → 0.1149
  DoS     : Brier 0.0502 → 0.0458 ↓   ECE 0.0368 → 0.0115
  Probe   : Brier 0.0348 → 0.0267 ↓   ECE 0.0440 → 0.0221
  R2L     : Brier 0.1517 → 0.1443 ↓   ECE 0.1030 → 0.0961
  U2R     : Brier 0.0130 → 0.0053 ↓   ECE 0.0285 → 0.0079

Calibrating xgb_5class_smote (5class) with Dirichlet ODIR...
  Normal  : Brier 0.1068 → 0.1046 ↓   ECE 0.1240 → 0.1190
  DoS     : Brier 0.0521 → 0.0459 ↓   ECE 0.0282 → 0.0112
  Probe   : Brier 0.0352 → 0.0273 ↓   ECE 0.0366 → 0.0205
  R2L     : Brier 0.1535 → 0.1419 ↓   ECE 0.1121 → 0.1040
  U2R

## 5. Pre-cal vs Dirichlet comparison (hybrid added later if available)

In [ ]:
# Try to load hybrid summary if it exists (from a UNSW hybrid notebook)
hybrid_summary = None
hybrid_path = CALIB_DIR_HYBRID / 'calibration_summary.json'
if hybrid_path.exists():
    with open(hybrid_path) as f:
        hybrid_summary = json.load(f)
    print(f'✓ Hybrid summary found: {hybrid_path}')
else:
    print(f'⚠ Hybrid summary not found at {hybrid_path}')
    print(f'  (UNSW hybrid notebook 03 not yet run — comparison will be pre vs Dirichlet only)')

rows = []
for name, task in ALL_MODELS:
    d = calibration_summary[name]
    d_pre = d['brier_pre']
    d_post = d['brier_post']

    if task == '5class':
        b_pre = np.mean([d_pre[c] for c in range(5)])
        b_dirichlet = np.mean([d_post[c] for c in range(5)])
    else:
        b_pre = d_pre[1]
        b_dirichlet = d_post[1]

    row = {
        'Model': name,
        'Task': '5-class' if task == '5class' else 'binary',
        'Brier pre':       round(b_pre,       5),
        'Brier dirichlet': round(b_dirichlet, 5),
        'Dirichlet delta': round(b_dirichlet - b_pre, 5),
    }

    if hybrid_summary is not None and name in hybrid_summary:
        h = hybrid_summary[name]
        h_post = h['brier_post']
        if task == '5class':
            b_hybrid = np.mean([h_post[str(c)] for c in range(5)])
        else:
            b_hybrid = h_post['1']
        row['Brier hybrid'] = round(b_hybrid, 5)

        methods = {'pre': b_pre, 'hybrid': b_hybrid, 'dirichlet': b_dirichlet}
        row['Winner'] = min(methods, key=methods.get)
    else:
        methods = {'pre': b_pre, 'dirichlet': b_dirichlet}
        row['Winner'] = min(methods, key=methods.get)

    rows.append(row)

df_cmp = pd.DataFrame(rows)
print('\nUNSW v2 — Calibration comparison (macro Brier)')
print('=' * 110)
print(df_cmp.to_string(index=False))
print('=' * 110)

TABLES_DIR = Path(REPO) / 'results' / 'tables'
csv_path = TABLES_DIR / 'unsw_v2_calibration_threeway.csv'
df_cmp.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

winners = df_cmp['Winner'].value_counts()
print(f'\nWinner counts: {winners.to_dict()}')

## 6. Per-class breakdown (5-class models only)

In [ ]:
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    d = calibration_summary[name]

    for c in range(5):
        d_pre = d['brier_pre'][c]
        d_post = d['brier_post'][c]

        row = {
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts[c],
            'Brier pre': round(d_pre, 5),
            'Brier dirichlet': round(d_post, 5),
            'Delta': round(d_post - d_pre, 5),
        }

        if hybrid_summary is not None and name in hybrid_summary:
            h = hybrid_summary[name]
            h_post = h['brier_post'][str(c)]
            h_strategy = h['strategies'][str(c)]
            row['Brier hybrid'] = round(h_post, 5)
            row['Hybrid strategy'] = h_strategy
            methods = {'pre': d_pre, 'hybrid': h_post, 'dirichlet': d_post}
        else:
            methods = {'pre': d_pre, 'dirichlet': d_post}
        row['Winner'] = min(methods, key=methods.get)

        perclass_rows.append(row)

df_perclass = pd.DataFrame(perclass_rows)
print('UNSW v2 — Per-class comparison')
print('=' * 140)
print(df_perclass.to_string(index=False))
print('=' * 140)

csv_path = TABLES_DIR / 'unsw_v2_calibration_threeway_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

print(f'\nWinners by class:')
for c in range(5):
    cname = CLASS_NAMES_5[c]
    cnt = Counter(df_perclass[df_perclass['Class'] == cname]['Winner'])
    print(f'  {cname:8s}: {dict(cnt)}')

## 7. Sanity checks

In [ ]:
print('Sanity checks on Dirichlet calibrated probabilities:')
print('-' * 80)

for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy')
    expected_cols = 2 if task == 'binary' else 5

    shape_ok = p_cal.shape[1] == expected_cols
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    finite_ok = np.isfinite(p_cal).all()

    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/03c_unsw_calibration_dirichlet_v2.ipynb
!git add calibrators/unsw_nb15_v2_dirichlet/
!git add results/tables/unsw_v2_calibration_threeway.csv
!git add results/tables/unsw_v2_calibration_threeway_perclass.csv
!git status --short
!git commit -m 'Notebook 03c-UNSW v2: Dirichlet calibration on balanced calib/test distribution'
!git push origin main